# Sampling 1.000 Data Positif dari `comments_preprocessed`

Menggunakan **Logistic Regression** (dengan class weights + CrossValidator)
untuk memprediksi komentar paling mungkin positif dari data yang belum dilabeli.
Data yang sudah ada di `comments_sentiment` di-exclude.
Hasil disimpan ke CSV lokal.


## 1. Import & Konfigurasi


In [1]:
import csv
import os
from pathlib import Path

from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer, IDF, NGram, RegexTokenizer,
    StringIndexer, StringIndexerModel, VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import FloatType


In [2]:
TEXT_COL = "text_final"
LABEL_COL = "sentiment"
VALID_LABELS = ("positif", "netral", "negatif")
SEED = 42
TARGET_SAMPLE = 1000

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(dotenv_path=PROJECT_ROOT / ".env", override=True)
MONGO_URI = os.getenv("MONGO_URI", "").strip()
MONGO_DB = os.getenv("MONGO_DB", "analisis_sentimen").strip()

MONGO_LABELED_COLLECTION = "comments_sentiment"
MONGO_PREPROC_COLLECTION = "comments_preprocessed"
OUTPUT_DIR = PROJECT_ROOT / "hasil"
OUTPUT_FILE = OUTPUT_DIR / "sampled_positif_1000_lr.csv"

print(f"Target   : {TARGET_SAMPLE} komentar positif")
print(f"Output   : {OUTPUT_FILE}")


Target   : 1000 komentar positif
Output   : D:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\hasil\sampled_positif_1000_lr.csv


In [3]:
def create_spark_session(app_name: str = "sample-positif-lr") -> SparkSession:
    java_home = os.environ.get("JAVA_HOME") or r"C:\\Program Files\\Java\\jdk-22"
    os.environ["JAVA_HOME"] = java_home
    os.environ["PYSPARK_PYTHON"] = os.environ.get("PYSPARK_PYTHON") or "python"
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.driver.memory", "8g") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.default.parallelism", "8") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .getOrCreate()
    spark._jsc.hadoopConfiguration().set(
        "fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem"
    )
    return spark


spark = create_spark_session()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} siap")


Spark 4.1.2 siap


## 2. Helper Functions


In [4]:
def build_pipeline(model_name: str, **kwargs):
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol="tokens",
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol="label_index", handleInvalid="keep"
    )
    ngram = NGram(n=2, inputCol="tokens", outputCol="bigrams")
    name = model_name.lower().strip()
    vocab_uni = kwargs.get("vocab_uni", 8000)
    vocab_bi = kwargs.get("vocab_bi", 6000)
    min_df = kwargs.get("min_df", 3.0)
    cv_uni = CountVectorizer(
        inputCol="tokens", outputCol="uni_feat",
        vocabSize=vocab_uni, minDF=min_df, minTF=1,
    )
    cv_bi = CountVectorizer(
        inputCol="bigrams", outputCol="bi_feat",
        vocabSize=vocab_bi, minDF=min_df, minTF=1,
    )
    assembler = VectorAssembler(
        inputCols=["uni_feat", "bi_feat"], outputCol="raw_feat"
    )
    idf = IDF(inputCol="raw_feat", outputCol="features", minDocFreq=2)
    clf = LogisticRegression(
        featuresCol="features", labelCol="label_index", predictionCol="pred_index",
        maxIter=kwargs.get("max_iter", 300), regParam=kwargs.get("reg_param", 0.05),
        elasticNetParam=kwargs.get("elastic_net", 0.15), family="multinomial", tol=1e-4,
    )
    stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]
    return Pipeline(stages=stages)


def map_prediction_labels(df, si_model):
    labels = list(si_model.labels)
    when_expr = None
    for i, lbl in enumerate(labels):
        cond = F.col("pred_index").cast("int") == i
        when_expr = F.when(cond, F.lit(lbl)) if when_expr is None else when_expr.when(cond, F.lit(lbl))
    return df.withColumn("pred_label", when_expr)


## 3. Load Labeled Data & Train LR


In [5]:
# Load data berlabel dari comments_sentiment
def _normalize_value(v):
    if v is None:
        return ''
    if isinstance(v, (int, float)):
        return str(v)
    return str(v)

train_docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_LABELED_COLLECTION].find(
        {TEXT_COL: {"$exists": True, "$nin": ["", None]},
         LABEL_COL: {"$exists": True, "$nin": [None, ""]}},
        {"_id": 0, "comment_id": 1, TEXT_COL: 1, LABEL_COL: 1},
    )
    for doc in cursor:
        train_docs.append({
            "comment_id": doc.get("comment_id", ""),
            TEXT_COL: _normalize_value(doc[TEXT_COL]),
            LABEL_COL: doc.get(LABEL_COL, ""),
        })

print(f"Data train: {len(train_docs)}")
train_df = spark.createDataFrame(train_docs).cache()
train_df.printSchema()
train_df.show(3, truncate=40)

Data train: 2000
root
 |-- comment_id: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- text_final: string (nullable = true)

+--------------------------+---------+----------------------------------------+
|                comment_id|sentiment|                              text_final|
+--------------------------+---------+----------------------------------------+
|UgzItBvbEwKAn1qfskh4AaABAg|  negatif|tonton paham seluruh omong pandji leb...|
|UgxS1lPEFSMZRhpMieJ4AaABAg|  negatif|usia tahun rusuh 1998 adik adik kecil...|
|Ugysox1PNUHnPNV5GLx4AaABAg|  negatif|sedang gusar sadar diri sudah mulai r...|
+--------------------------+---------+----------------------------------------+
only showing top 3 rows


In [6]:
# Class weights
label_counts = train_df.groupBy(LABEL_COL).count().collect()
total = sum(r["count"] for r in label_counts)
n_class = len(label_counts)
weight_dict = {r[LABEL_COL]: total / (n_class * r["count"]) for r in label_counts}
print("Class weights:")
for k, v in weight_dict.items():
    print(f"  {k} -> {v:.4f}")
mapping = F.create_map(
    *[x for kv in weight_dict.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
)
train_df_w = train_df.withColumn("class_weight", mapping[F.col(LABEL_COL)])


Class weights:
  positif -> 8.3333
  negatif -> 0.8150
  netral -> 0.6050


In [7]:
# Fit feature pipeline + LR dengan CrossValidator
pipeline_obj = build_pipeline("lr")
stages = pipeline_obj.getStages()
feat_stages = stages[:-1]
feat_model = Pipeline(stages=feat_stages).fit(train_df_w)
train_feat = feat_model.transform(train_df_w).cache()

base_lr = stages[-1]
base_lr.setWeightCol("class_weight")

param_grid = ParamGridBuilder() \
    .addGrid(base_lr.regParam, [0.01, 0.05, 0.1]) \
    .addGrid(base_lr.elasticNetParam, [0.0, 0.15, 0.5]) \
    .build()
evaluator = MulticlassClassificationEvaluator(
    labelCol='label_index', predictionCol='pred_index', metricName='f1'
)
cv = CrossValidator(
    estimator=base_lr, estimatorParamMaps=param_grid,
    evaluator=evaluator, numFolds=3, seed=SEED, parallelism=4,
)
cv_model = cv.fit(train_feat)
best_lr = cv_model.bestModel
print(f"Best LR: regParam={best_lr.getRegParam():.4f}, elasticNet={best_lr.getElasticNetParam():.4f}")


Best LR: regParam=0.1000, elasticNet=0.1500


## 4. Load Unlabeled Data (Exclude Duplikat)


In [8]:
# Kumpulkan exclude IDs dari comments_sentiment
exclude_ids = set()
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    for doc in client[MONGO_DB][MONGO_LABELED_COLLECTION].find(
        {}, {"comment_id": 1}
    ):
        cid = doc.get("comment_id", "")
        if cid:
            exclude_ids.add(cid)
print(f"Exclude: {len(exclude_ids)} dari {MONGO_LABELED_COLLECTION}")


Exclude: 2000 dari comments_sentiment


In [9]:
# Load dari comments_preprocessed, skip exclude
unlabeled_docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_PREPROC_COLLECTION].find(
        {TEXT_COL: {"$exists": True, "$nin": ["", None]}},
        {"_id": 0, "comment_id": 1, "video_id": 1, "text_original": 1, TEXT_COL: 1},
    )
    for doc in cursor:
        cid = doc.get("comment_id", "")
        if cid in exclude_ids:
            continue
        tf = _normalize_value(doc.get(TEXT_COL))
        if not tf:
            continue
        unlabeled_docs.append({
            "comment_id": cid,
            "video_id": doc.get("video_id", ""),
            "text_original": _normalize_value(doc.get("text_original")),
            TEXT_COL: tf,
        })

print(f"Unlabeled (setelah exclude): {len(unlabeled_docs)}")


Unlabeled (setelah exclude): 11177


In [10]:
# Spark DF + dummy label untuk StringIndexer
unlabeled_df = spark.createDataFrame(unlabeled_docs)
unlabeled_df = unlabeled_df.withColumn(LABEL_COL, F.lit("netral"))
print(f"Unlabeled DF: {unlabeled_df.count()} rows")


Unlabeled DF: 11177 rows


## 5. Prediksi & Ambil Top 1000 Positif


In [11]:
# Feature transform + prediksi
unlabeled_feat = feat_model.transform(unlabeled_df).cache()
pred_df = best_lr.transform(unlabeled_feat)
si_model = feat_model.stages[-1]
pred_df = map_prediction_labels(pred_df, si_model)

# Distribusi prediksi
for r in pred_df.groupBy("pred_label").count().orderBy("pred_label").collect():
    print(f"  Predicted {r['pred_label']:10s}: {r['count']:6d}")


  Predicted negatif   :   1942
  Predicted netral    :   9144
  Predicted positif   :     91


In [12]:
# Ekstrak probabilitas positif
labels = list(si_model.labels)
print(f"Label order: {labels}")
positif_idx = labels.index("positif")
print(f"Positif index: {positif_idx}")

extract_prob = F.udf(lambda v: float(v[positif_idx]), FloatType())
pred_df = pred_df.withColumn("positif_prob", extract_prob(F.col("probability")))


Label order: ['netral', 'negatif', 'positif']
Positif index: 2


In [13]:
# Filter positif, sortir by confidence desc
positif_df = pred_df \
    .filter(F.col("pred_label") == "positif") \
    .select("comment_id", "video_id", "text_original", TEXT_COL, F.col("pred_label").alias(LABEL_COL), "positif_prob") \
    .orderBy(F.col("positif_prob").desc())

n_found = positif_df.count()
n_take = min(n_found, TARGET_SAMPLE)
print(f"Diprediksi positif: {n_found}")
print(f"Diambil: {n_take}")

sample_df = positif_df.limit(n_take)
sample_df.show(10, truncate=40)


Diprediksi positif: 91
Diambil: 91
+----------------------------------------+-----------+----------------------------------------+----------------------------------------+---------+------------+
|                              comment_id|   video_id|                           text_original|                              text_final|sentiment|positif_prob|
+----------------------------------------+-----------+----------------------------------------+----------------------------------------+---------+------------+
|              Ugzf3vaWgnLcnBuDkS14AaABAg|F6fgLwUeeqI|Yang menolak ruu TNI dan demo fix mer...|tolak ruu_tni demo bodoh emo_other em...|  positif|   0.9899086|
|              UgwRndVokPB_34gJszB4AaABAg|sg8Mzx0fZbU|Kalo saya sih sangat setuju sekali bi...|sangat setuju sekali bila tni ambil k...|  positif|   0.8513523|
|              UgzHDU30VPRSMo37hhl4AaABAg|7CLZkPwhEG4|Sepertinya dari dulu inilah yg di tak...|takut kelompok senjata merdeka diri g...|  positif|  0.80828077|
|    

## 6. Simpan ke CSV


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["comment_id", "video_id", "text_original", "text_final", "sentiment"])
    for row in sample_df.collect():
        writer.writerow([
            row["comment_id"],
            row["video_id"],
            row["text_original"],
            row[TEXT_COL],
            row[LABEL_COL],
        ])

print(f"\nDisimpan ke: {OUTPUT_FILE}")
print(f"Total: {n_take} baris")



Disimpan ke: D:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\hasil\sampled_positif_1000_lr.csv
Total: 91 baris


## 7. Cek Statistika


In [15]:
if n_found < TARGET_SAMPLE:
    print(f"PERINGATAN: Hanya {n_found} diprediksi positif (< {TARGET_SAMPLE})")

probs = [r["positif_prob"] for r in sample_df.collect()]
print(f"\nStatistik probabilitas positif:")
print(f"  Min   : {min(probs):.4f}")
print(f"  Max   : {max(probs):.4f}")
print(f"  Mean  : {sum(probs)/len(probs):.4f}")


PERINGATAN: Hanya 91 diprediksi positif (< 1000)

Statistik probabilitas positif:
  Min   : 0.3766
  Max   : 0.9899
  Mean  : 0.5582


In [16]:
spark.stop()
print("\nSelesai.")



Selesai.
